In [1]:
1+1

2

# ACTS Merge and Vertex Smearing Testing

This notebook tests whether ACTS merged and smeared HepMC3 files genuinely have non-zero **PRIMARY** vertices after the merging and smearing process.

**Important:** This analysis focuses specifically on **PRIMARY vertices** - the initial interaction points where hard scatter and pileup events occur. We do NOT analyze secondary vertices from particle decays.

**Key Discovery:** HepMC3 and pyhepmc have **native vertex status codes**! According to the NuHepMC specification:
- **Status 1** = Primary vertex (initial interaction) ← This is what we want!
- **Status 2** = FSI Summary vertex  
- Other status codes = various physics processes

This is much more efficient than complex heuristics based on particle status codes.

**Test Goals:**
1. Verify that vertex smearing is applied correctly to PRIMARY vertices by ACTS HepMC3Reader
2. Compare primary vertex distributions before and after merging/smearing  
3. Validate that vertex smearing parameters are applied as expected to initial collision points
4. Check for proper primary vertex position distributions in x, y, z, and time coordinates

**Expected Results:**
- Original files should have primary vertices at or near (0,0,0,0)
- Merged/smeared files should show Gaussian distributions around (0,0,0,0) with sigmas matching configuration
- Only ~1-10% of vertices should be primary vertices (rest are secondary/decay vertices)


In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import warnings

# HepMC3 reading
import pyhepmc as hep

# Visualization
import atlasify as atl
atl.ATLAS = "ColliderML"

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')


In [3]:
def is_primary_vertex(vertex, event, first_vertex_cache=None):
    """
    Determine if a vertex is a primary vertex (initial interaction point).
    
    Primary vertices are characterized by:
    1. Having incoming particles with status -1 (beam particles)
    2. Being the initial interaction vertex in the event (fallback)
    
    Parameters:
    -----------
    vertex : pyhepmc.GenVertex
        The vertex to check
    event : pyhepmc.GenEvent
        The event containing the vertex
    first_vertex_cache : dict, optional
        Cache for first vertex per event to avoid repeated computation
        
    Returns:
    --------
    bool
        True if this is a primary vertex
    """
    # Method 1: Check if vertex has incoming beam particles (status -1)
    for particle in vertex.particles_in:
        if particle.status == -1:  # Beam particle
            return True, particle.status

    # Method 2: Check if this vertex has only outgoing particles (no incoming except beams)
    # This is often characteristic of primary vertices
    if len(vertex.particles_in) == 0:  # No incoming particles
        return True, -2
    
    return False, -2

def extract_primary_vertices_from_hepmc(file_path, max_events=None, debug=False):
    """
    Extract PRIMARY vertex positions from a HepMC3 file.
    
    This function identifies and extracts only the primary vertices (initial interaction points)
    where vertex smearing should be applied, not all vertices in the event.
    
    Parameters:
    -----------
    file_path : Path or str
        Path to the HepMC3 file
    max_events : int, optional
        Maximum number of events to read (None for all)
    debug : bool
        Print debugging information
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: event_id, vertex_id, x, y, z, t, is_primary
    """
    vertices_data = []
    
    try:
        if debug:
            print(f"Opening file: {file_path}")
            
        with hep.open(str(file_path)) as f:
            if debug:
                print("File opened successfully, starting event loop...")
                
            for event_idx, event in enumerate(f):
                if max_events and event_idx >= max_events:
                    break
                
                if debug and event_idx % 10 == 0:
                    print(f"Processing event {event_idx}...")
                
                try:
                    # Count total vertices first
                    total_vertices = len(list(event.vertices))
                    if debug and event_idx < 3:
                        print(f"  Event {event_idx}: {total_vertices} total vertices")
                    
                    primary_vertex_count = 0
                    
                    for vertex_idx, vertex in enumerate(event.vertices):
                        if debug and event_idx < 2 and vertex_idx < 5:
                            print(f"    Checking vertex {vertex_idx}...")
                            
                        is_primary = is_primary_vertex(vertex, event)
                        
                        # Only record primary vertices
                        if is_primary:
                            if debug and event_idx < 3:
                                print(f"    Found primary vertex {primary_vertex_count} at ({vertex.position.x:.3f}, {vertex.position.y:.3f}, {vertex.position.z:.3f})")
                                
                            vertices_data.append({
                                'event_id': event_idx,
                                'vertex_id': vertex_idx,
                                'primary_vertex_id': primary_vertex_count,
                                'x': vertex.position.x,  # mm
                                'y': vertex.position.y,  # mm 
                                'z': vertex.position.z,  # mm
                                't': vertex.position.t,  # mm (light-distance)
                                'is_primary': True
                            })
                            primary_vertex_count += 1
                            
                except Exception as e:
                    if debug:
                        print(f"  Error processing event {event_idx}: {e}")
                    continue
                    
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()
    
    if debug:
        print(f"Extraction complete. Found {len(vertices_data)} primary vertices.")
    
    return pd.DataFrame(vertices_data)

def extract_all_vertices_from_hepmc(file_path, max_events=None):
    """
    Extract ALL vertex positions from a HepMC3 file for comparison.
    
    Parameters:
    -----------
    file_path : Path or str
        Path to the HepMC3 file
    max_events : int, optional
        Maximum number of events to read (None for all)
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: event_id, vertex_id, x, y, z, t, is_primary
    """
    vertices_data = []
    
    try:
        with hep.open(str(file_path)) as f:
            for event_idx, event in enumerate(f):
                if max_events and event_idx >= max_events:
                    break
                    
                for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
                    is_primary, status = is_primary_vertex(vertex, event)
                    
                    vertices_data.append({
                        'event_id': event_idx,
                        'vertex_id': vertex_idx,
                        'x': vertex.position.x,  # mm
                        'y': vertex.position.y,  # mm 
                        'z': vertex.position.z,  # mm
                        't': vertex.position.t,  # mm (light-distance)
                        'is_primary': is_primary,
                        "particle_in_status": status,
                    })
                    
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()
    
    return pd.DataFrame(vertices_data)

def load_and_analyze_primary_vertices(file_path, label, max_events=100):
    """
    Load PRIMARY vertices from HepMC3 file and compute basic statistics.
    
    This function focuses specifically on primary vertices (initial interaction points)
    where vertex smearing should be applied for hard scatter and pileup events.
    
    Parameters:
    -----------
    file_path : Path or str
        Path to the HepMC3 file
    label : str
        Label for this dataset
    max_events : int
        Maximum number of events to analyze
        
    Returns:
    --------
    pd.DataFrame, dict
        Primary vertices DataFrame and summary statistics
    """
    print(f"\n=== Analyzing PRIMARY vertices in {label} ===")
    print(f"File: {file_path}")
    
    if not Path(file_path).exists():
        print(f"WARNING: File does not exist: {file_path}")
        return pd.DataFrame(), {}
    
    # Use debug mode for acts_merged to help diagnose hanging
    debug_mode = "acts_merged" in label.lower() or "merged" in label.lower()
    
    if debug_mode:
        print("🐛 Debug mode enabled for merged file analysis...")
        # Use fewer events for debugging
        debug_max_events = min(10, max_events)
        print(f"🐛 Limiting to {debug_max_events} events for debugging")
    else:
        debug_max_events = max_events
    
    # Extract primary vertices only
    print(f"Extracting primary vertices...")
    primary_vertices_df = extract_primary_vertices_from_hepmc(
        file_path, debug_max_events, debug=debug_mode
    )
    
    # Also extract all vertices for comparison (but limit if debugging)
    print(f"Extracting all vertices for comparison...")
    all_vertices_df = extract_all_vertices_from_hepmc(file_path, debug_max_events)
    
    if primary_vertices_df.empty:
        print("No primary vertices found!")
        return primary_vertices_df, {}
    
    # Calculate statistics for primary vertices
    stats = {
        'n_events': primary_vertices_df['event_id'].nunique(),
        'n_primary_vertices': len(primary_vertices_df),
        'n_total_vertices': len(all_vertices_df),
        'primary_vertices_per_event': len(primary_vertices_df) / primary_vertices_df['event_id'].nunique(),
        'total_vertices_per_event': len(all_vertices_df) / all_vertices_df['event_id'].nunique(),
        'x_mean': primary_vertices_df['x'].mean(),
        'x_std': primary_vertices_df['x'].std(),
        'y_mean': primary_vertices_df['y'].mean(), 
        'y_std': primary_vertices_df['y'].std(),
        'z_mean': primary_vertices_df['z'].mean(),
        'z_std': primary_vertices_df['z'].std(),
        't_mean': primary_vertices_df['t'].mean(),
        't_std': primary_vertices_df['t'].std(),
        'r_mean': np.sqrt(primary_vertices_df['x']**2 + primary_vertices_df['y']**2).mean(),
        'r_std': np.sqrt(primary_vertices_df['x']**2 + primary_vertices_df['y']**2).std(),
    }
    
    print(f"Events: {stats['n_events']}")
    print(f"PRIMARY vertices: {stats['n_primary_vertices']} ({stats['primary_vertices_per_event']:.1f} per event)")
    print(f"Total vertices: {stats['n_total_vertices']} ({stats['total_vertices_per_event']:.1f} per event)")
    print(f"Primary vertex fraction: {100*stats['n_primary_vertices']/stats['n_total_vertices']:.1f}%")
    print(f"Position means: x={stats['x_mean']:.3f}, y={stats['y_mean']:.3f}, z={stats['z_mean']:.3f}, t={stats['t_mean']:.3f} mm")
    print(f"Position stds:  x={stats['x_std']:.3f}, y={stats['y_std']:.3f}, z={stats['z_std']:.3f}, t={stats['t_std']:.3f} mm")
    print(f"Radial: mean={stats['r_mean']:.3f}, std={stats['r_std']:.3f} mm")
    
    return primary_vertices_df, stats


In [ ]:
# =============================================================================
# CONFIGURATION - Update these paths for your test files
# =============================================================================

# Base directory containing your simulation outputs
base_dir = Path("/global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v1/runs/0")

# Example file paths - update these to your actual files
file_paths = {
    # Before merging/smearing
    "signal_only": base_dir / "events.hepmc3",
    "pileup_only": base_dir / "events_pileup.hepmc3", 
    
    # After ACTS merging and smearing
    "acts_merged": base_dir / "merged_events.hepmc3",
}

# Expected smearing parameters (in mm) - should match your configuration
expected_sigmas = {
    'xy': 0.0125,  # Typical beam spot x/y sigma in mm
    'z': 55.5,    # Typical beam spot z sigma in mm  
    't': 5.0      # Time smearing sigma in ns (often 0)
}

# Analysis parameters
max_events_to_analyze = 1  # Limit for faster analysis

In [ ]:
print("Configuration:")
print(f"Base directory: {base_dir}")
print(f"Expected sigmas: {expected_sigmas}")
print(f"Max events to analyze: {max_events_to_analyze}")
print(f"\nConfigured files:")
for label, path in file_paths.items():
    exists = "✓" if path.exists() else "✗" 
    print(f"  {label}: {exists} {path}")

# Check if we have at least one file to analyze
available_files = {k: v for k, v in file_paths.items() if v.exists()}
if not available_files:
    print("\n⚠️  WARNING: No configured files exist! Please update the file paths above.")
else:
    print(f"\n✓ Found {len(available_files)} files to analyze.")


Configuration:
Base directory: /global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v1/runs/0
Expected sigmas: {'xy': 0.0125, 'z': 55.5, 't': 5.0}
Max events to analyze: 1

Configured files:
  signal_only: ✓ /global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v1/runs/0/events.hepmc3
  pileup_only: ✓ /global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v1/runs/0/events_pileup.hepmc3
  acts_merged: ✓ /global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v1/runs/0/merged_events.hepmc3

✓ Found 3 files to analyze.


In [ ]:
max_events = 1
signal_vertex_df = extract_all_vertices_from_hepmc(file_paths["signal_only"], max_events=max_events)
# pileup_vertex_df = extract_all_vertices_from_hepmc(file_paths["pileup_only"], max_events=max_events)
# acts_merged_vertex_df = extract_all_vertices_from_hepmc(file_paths["acts_merged"], max_events=max_events)

print(signal_vertex_df.head())
# print(pileup_vertex_df.head())
# print(acts_merged_vertex_df.head())

Processing vertices: 100%|██████████| 433/433 [00:00<00:00, 90566.68it/s]

   event_id  vertex_id    x    y    z    t  is_primary  particle_in_status
0         0          0  0.0  0.0  0.0  0.0       False                  -2
1         0          1  0.0  0.0  0.0  0.0       False                  -2
2         0          2  0.0  0.0  0.0  0.0       False                  -2
3         0          3  0.0  0.0  0.0  0.0       False                  -2
4         0          4  0.0  0.0  0.0  0.0       False                  -2


In [ ]:
signal_vertex_df

,event_id,vertex_id,x,y,z,t,is_primary,particle_in_status
0,0,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,-2
1,0,1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,-2
2,0,2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,-2
3,0,3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,-2
4,0,4,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,-2
...,...,...,...,...,...,...,...,...
428,0,428,-5.920949e-07,5.838983e-07,2.939285e-06,3.070115e-06,False,-2
429,0,429,7.985624e-07,5.401639e-06,6.316399e-06,8.387386e-06,False,-2
430,0,430,-4.654308e-15,2.350128e-13,3.899729e-13,4.805407e-13,False,-2
431,0,431,-4.654308e-15,2.350128e-13,3.899729e-13,4.805407e-13,False,-2


In [ ]:
with hep.open(str(file_paths["signal_only"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            print("Vertex", vertex, vertex.status)
            print("Particles in", [(particle, particle.id) for particle in vertex.particles_in])
            print("Particles out", [(particle, particle.id) for particle in vertex.particles_out])

        break
    

Processing vertices:   0%|          | 0/433 [00:00<?, ?it/s]

Processing vertices: 100%|██████████| 433/433 [00:00<00:00, 13617.67it/s]

Vertex GenVertex(FourVector(0, 0, 0, 0)) 0
Particles in [(GenParticle(FourVector(0, 0, 6.5e+03, 6.5e+03), mass=0.93827, pid=2212, status=4), 1)]
Particles out [(GenParticle(FourVector(0, 0, 1.57e+03, 1.57e+03), mass=0, pid=2, status=61), 2), (GenParticle(FourVector(0, 0, 4.93e+03, 4.93e+03), mass=0.57933, pid=2101, status=63), 3)]
Vertex GenVertex(FourVector(0, 0, 0, 0)) 0
Particles in [(GenParticle(FourVector(0, 0, -6.5e+03, 6.5e+03), mass=0.93827, pid=2212, status=4), 4)]
Particles out [(GenParticle(FourVector(-1.89e-29, 0, -414, 414), mass=0, pid=21, status=61), 5), (GenParticle(FourVector(0, 0, -4.89e+03, 4.89e+03), mass=0.57933, pid=2101, status=63), 6), (GenParticle(FourVector(0, 0, -1.2e+03, 1.2e+03), mass=0.33, pid=2, status=63), 7)]
Vertex GenVertex(FourVector(0, 0, 0, 0)) 0
Particles in [(GenParticle(FourVector(0, 0, 1.57e+03, 1.57e+03), mass=0, pid=2, status=61), 2)]
Particles out [(GenParticle(FourVector(-1.09e-14, -1.07e-13, 1.57e+03, 1.57e+03), mass=0, pid=2, status=53), 

In [ ]:
with hep.open(str(file_paths["pileup_only"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            print("Vertex", vertex, vertex.status)
            print("Particles in", [(particle, particle.id) for particle in vertex.particles_in])
            print("Particles out", [(particle, particle.id) for particle in vertex.particles_out])

        break
    

Processing vertices: 100%|██████████| 374/374 [00:00<00:00, 17521.75it/s]

Vertex GenVertex(FourVector(0, 0, 0, 0)) 0
Particles in [(GenParticle(FourVector(0, 0, 7e+03, 7e+03), mass=0.93827, pid=2212, status=4), 1)]
Particles out [(GenParticle(FourVector(0.39, -0.796, 55.2, 55.2), mass=0, pid=-1, status=61), 2), (GenParticle(FourVector(0.356, -1.02, 712, 712), mass=0, pid=21, status=61), 3), (GenParticle(FourVector(0.41, 1.42, 85.1, 85.2), mass=0, pid=21, status=61), 4), (GenParticle(FourVector(-0.332, 1.18, 13.7, 13.7), mass=0, pid=2, status=61), 5), (GenParticle(FourVector(-0.219, 0.0895, 28.7, 28.7), mass=0, pid=21, status=61), 6), (GenParticle(FourVector(-0.397, 0.125, 1.38e+03, 1.38e+03), mass=0.77133, pid=2203, status=63), 7), (GenParticle(FourVector(0.597, -0.318, 805, 805), mass=0.33, pid=1, status=63), 8), (GenParticle(FourVector(-0.525, -0.231, 1.05e+03, 1.05e+03), mass=0.33, pid=1, status=63), 9), (GenParticle(FourVector(-0.279, -0.457, 2.87e+03, 2.87e+03), mass=0.33, pid=-2, status=63), 10)]
Vertex GenVertex(FourVector(0, 0, 0, 0)) 0
Particles in 

In [ ]:
with hep.open(str(file_paths["acts_merged"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            print("Vertex", vertex, vertex.status)
            print("Particles in", [(particle, particle.id) for particle in vertex.particles_in])
            print("Particles out", [(particle, particle.id) for particle in vertex.particles_out])

            if vertex_idx > 1000:
                break

        break
    

Processing vertices:   2%|▏         | 1001/41458 [00:00<00:01, 22953.78it/s]

Vertex GenVertex(FourVector(0.00282, 0.00663, -10.2, 1.97e+03)) 0
Particles in [(GenParticle(FourVector(0, 0, 6.5e+03, 6.5e+03), mass=0.93827, pid=2212, status=4), 1)]
Particles out [(GenParticle(FourVector(0, 0, 1.57e+03, 1.57e+03), mass=0, pid=2, status=61), 2), (GenParticle(FourVector(0, 0, 4.93e+03, 4.93e+03), mass=0.57933, pid=2101, status=63), 3)]
Vertex GenVertex(FourVector(0.00282, 0.00663, -10.2, 1.97e+03)) 0
Particles in [(GenParticle(FourVector(0, 0, -6.5e+03, 6.5e+03), mass=0.93827, pid=2212, status=4), 4)]
Particles out [(GenParticle(FourVector(-1.89e-29, 0, -414, 414), mass=0, pid=21, status=61), 5), (GenParticle(FourVector(0, 0, -4.89e+03, 4.89e+03), mass=0.57933, pid=2101, status=63), 6), (GenParticle(FourVector(0, 0, -1.2e+03, 1.2e+03), mass=0.33, pid=2, status=63), 7)]
Vertex GenVertex(FourVector(0.00282, 0.00663, -10.2, 1.97e+03)) 0
Particles in [(GenParticle(FourVector(0, 0, 1.57e+03, 1.57e+03), mass=0, pid=2, status=61), 2)]
Particles out [(GenParticle(FourVector(-

In [ ]:
vertex.status

0

In [ ]:
(vertex.position.x, vertex.position.y, vertex.position.z) == (0,0,0)

False

In [ ]:
zero_vertices_count = 0

with hep.open(str(file_paths["signal_only"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            if (vertex.position.x, vertex.position.y, vertex.position.z) == (0, 0, 0):
                zero_vertices_count += 1

        break
    
print(f"Number of zero vertices: {zero_vertices_count}")

Processing vertices: 100%|██████████| 433/433 [00:00<00:00, 288412.52it/s]

Number of zero vertices: 332


In [ ]:
zero_vertices_count = 0

with hep.open(str(file_paths["pileup_only"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            if (vertex.position.x, vertex.position.y, vertex.position.z) == (0, 0, 0):
                zero_vertices_count += 1

        break
    
print(f"Number of zero vertices: {zero_vertices_count}")

Processing vertices: 100%|██████████| 374/374 [00:00<00:00, 279620.27it/s]

Number of zero vertices: 287


In [ ]:
zero_vertices_count = 0

with hep.open(str(file_paths["acts_merged"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            if (vertex.position.x, vertex.position.y, vertex.position.z) == (0, 0, 0):
                zero_vertices_count += 1

        break
    
print(f"Number of zero vertices: {zero_vertices_count}")

Processing vertices: 100%|██████████| 41458/41458 [00:00<00:00, 315374.59it/s]

Number of zero vertices: 0


In [ ]:
from collections import defaultdict
unique_vertex_count = defaultdict(int)

with hep.open(str(file_paths["acts_merged"])) as f:
    for event_idx, event in enumerate(f):
        for vertex_idx, vertex in tqdm(enumerate(event.vertices), total=len(event.vertices), desc="Processing vertices"):
            
            unique_vertex_count[(vertex.position.x, vertex.position.y, vertex.position.z)] += 1

        break
    
print(f"Number of unique vertices: {len(unique_vertex_count)}")

Processing vertices: 100%|██████████| 41458/41458 [00:00<00:00, 281875.29it/s]

Number of unique vertices: 8871


In [ ]:
for vertex_key, vertex_count in unique_vertex_count.items():
    if vertex_count > 10:
        print(vertex_key, vertex_count)











(0.0028191632457433215, 0.006634132703706954, -10.213688787746822) 332
(0.002819163245739365, 0.006634132703705222, -10.213688787746825) 24
(-0.004089056988869888, 0.0238923126363892, -4.314154322098212) 287
(-0.015572488477822553, -0.009119560397659841, 32.75164750787284) 17
(0.00689569300829861, -0.019334116310905827, 15.104541931986017) 44
(0.007657316284888603, -0.007567662742466179, -28.289627210048153) 78
(-0.02034619038559945, 0.007971979390390268, 15.906105146787882) 89
(0.009275615362377558, 0.005411541387337255, -8.44889636898592) 109
(-0.007003758710742746, 0.008221784528645904, -39.77507759693987) 391
(-0.028068625754031924, -0.004144224624605063, 11.283810658465683) 393
(-0.004828448046545261, -0.005228663437317941, 10.132688325654524) 179
(-0.010666762403874087, -0.007830862327052662, -0.49866869660297203) 68
(0.0014983914872576647, 0.00947484140346928, 46.00479961772965) 136
(0.01526604450765099, -0.0034353091200496596, -24.7487129980009) 64
(-0.01203895300971856, -0.003

## EDM4Hep file testing

In [8]:
import uproot
from edm4hep_utils import load_edm4hep_file

In [11]:
edm4hep_path = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v1/runs/0/edm4hep.root"

In [ ]:
events = uproot.open(edm4hep_path)["events"]

NameError: name 'file_path' is not defined

In [ ]:
1+1